# Perbandingan Klasifikasi KNN vs Random Forest (GLCM + HOG)
Notebook ini membandingkan kinerja model **K-Nearest Neighbors (KNN)** dan **Random Forest** dalam mengklasifikasikan APD (Helm dan Kacamata) menggunakan ekstraksi fitur gabungan **GLCM (tekstur)** dan **HOG (bentuk)**.

In [ ]:
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from skimage.feature import graycomatrix, graycoprops, hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

IMG_SIZE = (128, 128)

def extract_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img_resized = cv2.resize(img, IMG_SIZE)
    distances = [1, 3]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    glcm = graycomatrix(img_resized, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)
    contrast      = graycoprops(glcm, 'contrast').flatten()
    correlation   = graycoprops(glcm, 'correlation').flatten()
    energy        = graycoprops(glcm, 'energy').flatten()
    homogeneity   = graycoprops(glcm, 'homogeneity').flatten()
    asm           = graycoprops(glcm, 'ASM').flatten()
    dissimilarity = graycoprops(glcm, 'dissimilarity').flatten()
    glcm_features = np.hstack([contrast, correlation, energy, homogeneity, asm, dissimilarity])
    hog_features  = hog(img_resized, orientations=8, pixels_per_cell=(16, 16),
                        cells_per_block=(2, 2), visualize=False)
    return np.hstack([glcm_features, hog_features])

def load_dataset(base_dir):
    features_list, labels_list = [], []
    classes = ['helm', 'kacamata']
    for class_idx, class_name in enumerate(classes):
        class_dir = os.path.join(base_dir, class_name)
        if not os.path.exists(class_dir):
            continue
        files = glob.glob(os.path.join(class_dir, '*.*'))
        print(f"Memproses kelas '{class_name}': {len(files)} gambar...")
        for img_path in tqdm(files, desc=f'Ekstraksi {class_name}'):
            feat = extract_features(img_path)
            if feat is not None:
                features_list.append(feat)
                labels_list.append(class_idx)
    return np.array(features_list), np.array(labels_list)

print('Memuat dataset training (GLCM + HOG)...')
X_raw, y_raw = load_dataset('dataset/train')
print(f'\nSelesai! Jumlah sampel: {X_raw.shape[0]}, Dimensi fitur: {X_raw.shape[1]}')


## Mencari Nilai K Optimal untuk KNN

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)
scaler_val   = StandardScaler()
X_tr_scaled  = scaler_val.fit_transform(X_tr)
X_val_scaled = scaler_val.transform(X_val)

k_values, val_accuracies = list(range(1, 16, 2)), []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_scaled, y_tr)
    acc = accuracy_score(y_val, knn.predict(X_val_scaled))
    val_accuracies.append(acc)
    print(f'  k = {k:2d}  ->  Akurasi Validasi: {acc*100:.2f}%')

plt.figure(figsize=(10, 5))
plt.plot(k_values, val_accuracies, marker='o', linestyle='-', color='#1e88e5', linewidth=2.5, markersize=8)
plt.title('Nilai K vs Akurasi Validasi (GLCM + HOG)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Nilai K', fontsize=12)
plt.ylabel('Akurasi Validasi', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.xticks(k_values)
for i, acc in enumerate(val_accuracies):
    plt.annotate(f'{acc:.4f}', (k_values[i], acc), textcoords='offset points',
                 xytext=(0,10), ha='center', fontsize=9, fontweight='bold')
plt.ylim(min(val_accuracies) - 0.05, max(val_accuracies) + 0.05)
plt.tight_layout()
plt.show()

best_k = k_values[int(np.argmax(val_accuracies))]
print(f'\nNilai K optimal: {best_k} (Akurasi Validasi: {max(val_accuracies)*100:.2f}%)')


## Pelatihan Model Final KNN & Random Forest

In [ ]:
scaler_final  = StandardScaler()
X_train_final = scaler_final.fit_transform(X_raw)

print(f'Melatih model KNN (k={best_k})...')
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_final, y_raw)

print('Melatih model Random Forest (100 trees)...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_final, y_raw)

print('Pelatihan kedua model selesai!')


## Evaluasi & Perbandingan pada Dataset Test

In [ ]:
print('Memuat dataset test...')
X_test_raw, y_test = load_dataset('dataset/test')
print(f'Selesai! Jumlah sampel test: {X_test_raw.shape[0]}')

X_test_scaled = scaler_final.transform(X_test_raw)
y_pred_knn = knn_model.predict(X_test_scaled)
y_pred_rf  = rf_model.predict(X_test_scaled)

acc_knn = accuracy_score(y_test, y_pred_knn)
acc_rf  = accuracy_score(y_test, y_pred_rf)

labels = ['Helm', 'Kacamata']

# Confusion Matrix berdampingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm_knn = confusion_matrix(y_test, y_pred_knn)
ConfusionMatrixDisplay(confusion_matrix=cm_knn, display_labels=labels).plot(ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix - KNN (k={best_k})\nAkurasi: {acc_knn*100:.2f}%')

cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(confusion_matrix=cm_rf, display_labels=labels).plot(ax=axes[1], cmap='Greens')
axes[1].set_title(f'Confusion Matrix - Random Forest\nAkurasi: {acc_rf*100:.2f}%')

plt.tight_layout()
plt.show()

print('\n--- CLASSIFICATION REPORT KNN ---')
print(classification_report(y_test, y_pred_knn, target_names=labels, zero_division=0))

print('\n--- CLASSIFICATION REPORT RANDOM FOREST ---')
print(classification_report(y_test, y_pred_rf, target_names=labels, zero_division=0))

# Grafik perbandingan akurasi
plt.figure(figsize=(6, 4))
bars = plt.bar(['KNN', 'Random Forest'], [acc_knn * 100, acc_rf * 100],
               color=['#1e88e5', '#43a047'], width=0.4)
plt.ylabel('Akurasi (%)')
plt.title('Perbandingan Akurasi KNN vs Random Forest (GLCM + HOG)')
plt.ylim(0, 110)
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
             f'{height:.2f}%', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
